## RAG Build

This notebook is the test build for the RAG, the feature complete version is in rag_evaluate.ipynb

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import config


In [2]:
from pathlib import Path
from pypdf import PdfReader
from langchain_core.documents import Document



# Load Documents method
def load_documents(file_path: str) -> list[Document]:
    """
    Loads Every PDF in a directory as page-level LangChain documents
    """
    documents = []

    for pdf_path in Path(file_path).glob("*pdf"):
        reader = PdfReader(pdf_path)

        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text()

            if text and text.strip():
                documents.append(
                    Document(
                        page_content=text.strip(),
                        metadata={
                            "source":pdf_path.name,
                            "page": page_number
                        }
                    )
                )
    if not documents:
        raise ValueError(f"No readable PDFs found in {file_path}")

    return documents

# Loads the F1 Regulations
docs = load_documents(config.RAW_DATA_DIR)
print(f"Loaded {len(docs)} pages")
print(docs[0].metadata)
print(docs[0].page_content[:500])


Loaded 596 pages
{'source': 'financial_regulations-f1_teams.pdf', 'page': 1}
SECTION D: FINANCIAL REGULATIONS (F1 TEAMS) 
 
 
 
 
0 
 
D 
 
  
D1 2026 Formula 1: Financial Regulations (F1 Teams) 
©2026 Fédération Internationale de l’Automobile 
25 June 2026  
Issue 07  
SECTION D: FINANCIAL REGULATIONS (F1 TEAMS) 
 
Version:   Issue 07 
Status:    PUBLISHED 
Date:    25/06/2026 
WMSC approval date:  23/06/2026 
 
CONVENTION: 
Black Text: Text unchanged from 2026 FIA F1 Regulations – Section D [Financial – F1 Teams] – Iss 06 
Pink Text: Changes relatives to 2026 FIA F1 Re


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split method
def split_documents(
        documents,
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP
    ):
    """
    Splits documents into a split list of documents based on chunk parameters
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    return text_splitter.split_documents(documents)

# Split the F1 Regulations
splits = split_documents(docs)

print(f"Created {len(splits)} chunks")
print(splits[0].metadata)
print(splits[0].page_content[:500])

c:\Users\thorp\Documents\GitHub\rag-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Created 2462 chunks
{'source': 'financial_regulations-f1_teams.pdf', 'page': 1}
SECTION D: FINANCIAL REGULATIONS (F1 TEAMS) 
 
 
 
 
0 
 
D 
 
  
D1 2026 Formula 1: Financial Regulations (F1 Teams) 
©2026 Fédération Internationale de l’Automobile 
25 June 2026  
Issue 07  
SECTION D: FINANCIAL REGULATIONS (F1 TEAMS) 
 
Version:   Issue 07 
Status:    PUBLISHED 
Date:    25/06/2026 
WMSC approval date:  23/06/2026 
 
CONVENTION: 
Black Text: Text unchanged from 2026 FIA F1 Regulations – Section D [Financial – F1 Teams] – Iss 06 
Pink Text: Changes relatives to 2026 FIA F1 Re


In [4]:
# Embed and Retrieve

from src import config
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="f1_regulations",
    persist_directory=str(config.VECTOR_STORE_DIR),
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": config.TOP_K}
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13716.45it/s]


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an assistant that answers questions about the 2026 Formula 1 regulations.

            Answer using the provided context.
            If the answer is not in the context, say:
            "I could not find that information in the indexed documents."

            Ensure that you include a reference to the regulation section and page number.

            Context:
            {context}
            """,
        ),
        ("human", "{question}"),
    ]
)

rag_chain = prompt | llm

In [6]:
# Retriever Test

question = "What happens when a race is suspended?"

retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

response = rag_chain.invoke(
    {
        "context": context,
        "question": question,
    }
)

print(response.content)

According to Article B5.14.2, if a race is suspended, the duration of the suspension will be added to this two (2) hour period up to a maximum total Race duration of three (3) hours.

Additionally, if the leader crosses the line at the end of the lap following the lap during which the sum of these periods ended, they will be shown the end-of-session signal. However, this does not result in exceeding the scheduled number of laps.

If the formation lap for the race is started behind the Safety Car (Article B5.10), the maximum total Race duration of three (3) hours will commence at the time the green lights on the start line are shown.


In [7]:
# Retriever Test

question = "What is the difference between the constructors championship, and the drivers championship?"

retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

response = rag_chain.invoke(
    {
        "context": context,
        "question": question,
    }
)

print(response.content)

According to Article A2.1 of the 2026 Formula 1 regulations, "Constructors' Championship" means the FIA Formula One World Constructors’ Championship.

The Constructors' Championship is a title that F1 Teams (or Constructors) compete for during each edition of the Championship. The F1 Team (or Constructor) with the highest number of points at the end of the season, calculated in accordance with Article A2.2, wins the Constructors' Championship title.

On the other hand, "Drivers' Championship" refers to the individual title awarded to the driver who scores the most points during the season, as outlined in Article A3.1.

Therefore, the main difference between the two championships is that one is a team-based competition (Constructors' Championship), while the other is an individual competition (Drivers' Championship).


In [8]:
# Retriever Test

question = "How much money can teams spend this year?"

retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

response = rag_chain.invoke(
    {
        "context": context,
        "question": question,
    }
)

print(response.content)

According to regulation D4.1.3, for a Full Year Financial Regulations Reporting Period ending on 31 December 2026, assuming 24 Competitions or fewer take place in that period, the Cost Cap is as follows:

- US Dollars: $215,000,000
- Pounds Sterling: £170,090,000
- Euros: €198,663,000
- Swiss Francs: CHF 189,992,000

These amounts are adjusted for Indexation if applicable.


In [9]:
# Retriever Test

question = "How many cameras do the cars have?"

retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

response = rag_chain.invoke(
    {
        "context": context,
        "question": question,
    }
)

print(response.content)

According to regulation C8.16 Cameras and camera housings, all cars must carry six positions in which cameras or camera housings can be fitted.

In positions 4 and 5, a single camera is required. In positions 1 and 2, a camera or camera housing is required. 

Additionally, if requested by the Commercial Rights Holder, a car may also carry an additional lightweight camera housing or diagnostic camera in position 3 (not mentioned in your query but part of the regulation).

Therefore, all cars must have at least 4 cameras (in positions 1, 2, 4 and 5) and one more camera or camera housing in either position 1, 2, or 3.
